Packages

In [29]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.stats import norm

## Méthode de Monte-Carlo pour estimer le prix d'une option 

On simule un grand nombre de trajectoires du sous-jacent, on calcule le payoff à maturité pour chacune, puis on prend la moyenne actualisée. Par la LFGN, cette moyenne converge vers le prix théorique de l'option.

In [30]:
def Monte_carlo_pricer(S0, K, sigma, r, q, T, N, nb_simu, option_type='call'):
    dt = T / N
    
    # Simulation des trajectoires
    epsilon = np.random.randn(N, nb_simu)
    dz = epsilon * np.sqrt(dt)
    dG = (r - q - (sigma**2)/2) * dt + sigma * dz
    
    variation = np.sum(dG, axis=0)
    St = S0 * np.exp(variation)
    
    # Calcul du payoff selon le type d'option
    if option_type == 'call':
        pay_off = np.maximum(St - K, 0)
    elif option_type == 'put':
        pay_off = np.maximum(K - St, 0)
        
    # Espérance du payoff
    pay_off_mean = np.mean(pay_off)
    
    # Actualisation du prix de l'option
    price = pay_off_mean * np.exp(-r * T)
    
    return price

Prix d'une option d'après Balack-Scholes

In [34]:
def black_scholes_pricer(S0, K, sigma, r, q, T, option_type='call'):
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == 'put':
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * np.exp(-q * T) * norm.cdf(-d1)
    return price

Comparaison des deux prix

In [33]:
# Paramètres
# S0 : prix initial du sous-jacent
# K : prix d'exercice de l'option
# sigma : volatilité du sous-jacent
# r : taux d'intérêt sans risque
# q : taux de dividende du sous-jacent
# T : maturité de l'option en années
# N : nombre de pas de temps dans la simulation
# nb_simu : nombre de simulations à effectuer pour la méthode de Monte Carlo
S0 = 130
K = 135
sigma = 0.3
r = 0.05
q = 0.02
T = 0.5
N = 1000
nb_simu = 100000
option_type = 'call'

prix_monte_carlo = Monte_carlo_pricer(S0, K, sigma, r, q, T, N, nb_simu, option_type)
prix_bs = black_scholes_pricer(S0, K, sigma, r, q, T, option_type)

diff = np.abs(prix_monte_carlo - prix_bs)/prix_bs * 100

print("le prix du call obtenu avec la méthode de Monte-Carlo est : ", prix_monte_carlo)
print("le prix du call théorique (Black-Scholes) est : ", prix_bs)
print("la différence entre les deux méthodes est de : ", diff, "%")



le prix du call obtenu avec la méthode de Monte-Carlo est :  9.615348276110618
le prix du call théorique (Black-Scholes) est :  9.57957631012561
la différence entre les deux méthodes est de :  0.3734190827124267 %
